In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
#Load fine-tuned model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "FatalHub/qwen-function-calling-10k"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/1.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.03k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [3]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio
import json
import re
import threading
import nest_asyncio
import torch

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok

In [4]:
tools = [
    {
        "name": "get_stock_price",
        "description": "Get the current stock price and daily change for a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_historical_prices",
        "description": "Get daily historical stock prices for a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_company_info",
        "description": "Get general company information and financial metrics.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_company_news",
        "description": "Get recent news articles and sentiment information for a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_exchange_rate",
        "description": "Get the current exchange rate between two currencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "from_currency": {
                    "type": "string",
                    "description": "The three-letter currency code to convert from, e.g. USD, EUR, GBP."
                },
                "to_currency": {
                    "type": "string",
                    "description": "The three-letter currency code to convert to, e.g. USD, EUR, GBP."
                }
            },
            "required": ["from_currency", "to_currency"]
        }
    }
]

In [17]:
def generate_tool_call(query, tools, max_new_tokens=256):
    
    tools_json = json.dumps(tools, indent=2)

    prompt = f"""You are a tool-calling assistant.

Tools:
{json.dumps(tools, indent=2)}

Rules:
1. You may ONLY use tools listed above.
2. If none of the available tools can answer the query, do NOT call any tool.
3. Do NOT invent tool names.
4. Do NOT use a tool for an indirect or unrelated purpose.
5. If no suitable tool exists, answer naturally that no suitable tool is available.
6. Do not explain how to use a tool.
7. For a valid tool call, return only the JSON tool call.

Query:
{query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK


In [12]:
def parse_tool_call(response):
    response = response.strip()

    try:
        result = json.loads(response)
    except json.JSONDecodeError:
        list_match = re.search(r'\[.*\]', response, re.DOTALL)

        if list_match:
            result = json.loads(list_match.group(0))
        else:
            obj_match = re.search(r'\{.*\}', response, re.DOTALL)

            if obj_match:
                result = json.loads(obj_match.group(0))
            else:
                raise ValueError(
                    f"Model did not return valid JSON:\n{response}"
                )

    if isinstance(result, dict):
        result = [result]

    if not isinstance(result, list):
        raise ValueError("Model output must be a list of tool calls.")

    return result

In [7]:
test_query = "What is microsoft's current stock price?"

response = generate_tool_call(
    test_query,
    tools
)

print(response)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{"name": "get_stock_price", "arguments": {"symbol": "MSFT"}}]


In [8]:
app = FastAPI(title="Fine-Tuned Tool Calling API")

class GenerateRequest(BaseModel):
    query: str
    tools: list

class GenerateResponse(BaseModel):
    tool_call: list

@app.post("/generate", response_model=GenerateResponse)
def generate(request: GenerateRequest):

    response = generate_tool_call(
        query=request.query,
        tools=request.tools
    )

    tool_call = parse_tool_call(response)

    return {
        "tool_call": tool_call
    }

In [9]:
nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

In [10]:
NGROK_AUTH_TOKEN = "3H8agUnb5HSsktvHRvitf6yeZQN_5wZ4bpCXmh8T3zZ1RRWGL"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)

public_url = public_url.public_url

print(public_url)

INFO:     Started server process [48]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


https://scalding-creme-broiling.ngrok-free.dev                                                      


In [16]:
import requests

payload = {
    "query": "What is Apple's current stock price? and tell me about the company",
    "tools": tools
}

response = requests.post(
    f"{public_url}/generate",
    json=payload
)

print(response.status_code)
print(response.json())

[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     34.21.232.215:0 - "POST /generate HTTP/1.1" 200 OK
200
{'tool_call': [{'name': 'get_stock_price', 'arguments': {'symbol': 'AAPL'}}, {'name': 'get_company_info', 'arguments': {'symbol': 'AAPL'}}]}
INFO:     197.120.248.93:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     197.120.248.93:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     197.120.248.93:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 422 Unprocessable Content
INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 422 Unprocessable Content
INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 422 Unprocessable Content


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 422 Unprocessable Content


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     197.120.248.93:0 - "POST /generate HTTP/1.1" 200 OK
